Script completo en Python para entrenar y evaluar un Árbol de Decisión comparando la Representación 1 (Z-score) contra la Representación 2 (Binning por Rangos) mediante Validación Cruzada Estratificada de 5 pliegues (5-Fold Stratified CV).

In [5]:
import pandas as pd
import numpy as np
from sklearn.model_selection import cross_validate, StratifiedKFold
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.preprocessing import StandardScaler

# ==============================================================================
# 1. CARGA Y PREPARACIÓN DEL DATASET (Valores del 1 al 10 imputados)
# ==============================================================================
feature_cols = [
    'Clump_thickness', 'Uniformity_of_cell_size', 'Uniformity_of_cell_shape',
    'Marginal_adhesion', 'Single_epithelial_cell_size', 'Bare_nuclei',
    'Bland_chromatin', 'Normal_nucleoli', 'Mitoses'
]

# Reemplaza 'breast_cancer.csv' por la ruta de tu archivo original
df = pd.read_csv('../data/breast_cancer_original.csv', names=(feature_cols + ['Class']), header=0)

# Limpieza e imputación previa
if 'Sample_code_number' in df.columns:
    df = df.drop(columns=['Sample_code_number'])

df['Bare_nuclei'] = pd.to_numeric(df['Bare_nuclei'].replace('?', np.nan))
df['Bare_nuclei'] = df['Bare_nuclei'].fillna(df['Bare_nuclei'].median())

# Normalizar la clase objetivo a 0 (Benigno) y 1 (Maligno)
y = df['Class'].map({2: 0, 4: 1}).values

# ==============================================================================
# 2. CONSTRUCCIÓN DE LAS DOS REPRESENTACIONES
# ==============================================================================
# Representación 1: Estandarización Z-Score
scaler = StandardScaler()
X_rep1 = scaler.fit_transform(df[feature_cols])

# Representación 2: Discretización por Rangos (Binning 0: Bajo, 1: Moderado, 2: Alto)
bins = [-np.inf, 3, 6, np.inf]
labels = [0, 1, 2]
X_rep2 = pd.DataFrame()
for col in feature_cols:
    X_rep2[col] = pd.cut(df[col], bins=bins, labels=labels).astype(int)

# ==============================================================================
# 3. EVALUACIÓN DE ÁRBOLES DE DECISIÓN (5-Fold Stratified CV)
# ==============================================================================
modelo_dt1 = DecisionTreeClassifier(max_depth=4, random_state=42)
modelo_dt2 = DecisionTreeClassifier(max_depth=4, random_state=42)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
metricas = ['accuracy', 'precision', 'recall', 'f1']

res_rep1 = cross_validate(modelo_dt1, X_rep1, y, cv=cv, scoring=metricas)
res_rep2 = cross_validate(modelo_dt2, X_rep2, y, cv=cv, scoring=metricas)

# ==============================================================================
# 4. TABLA DE RESULTADOS COMPARATIVOS
# ==============================================================================
df_resultados = pd.DataFrame({
    'Métrica': ['Exactitud (Accuracy)', 'Precisión (Precision)', 'Sensibilidad (Recall)', 'Puntuación F1 (F1-Score)'],
    'Rep. 1: Z-Score': [
        f"{res_rep1['test_accuracy'].mean():.4f} ± {res_rep1['test_accuracy'].std():.4f}",
        f"{res_rep1['test_precision'].mean():.4f}",
        f"{res_rep1['test_recall'].mean():.4f}",
        f"{res_rep1['test_f1'].mean():.4f}"
    ],
    'Rep. 2: Binning': [
        f"{res_rep2['test_accuracy'].mean():.4f} ± {res_rep2['test_accuracy'].std():.4f}",
        f"{res_rep2['test_precision'].mean():.4f}",
        f"{res_rep2['test_recall'].mean():.4f}",
        f"{res_rep2['test_f1'].mean():.4f}"
    ]
})

print("\n--- RESULTADOS COMPARATIVOS ---")
print(df_resultados.to_string(index=False))

# Imprimir las reglas del Árbol Discretizado para interpretación médica
modelo_dt2.fit(X_rep2, y)
print("\n--- REGLAS DE DECISIÓN GENERADAS (Representación 2: Binning) ---")
print(export_text(modelo_dt2, feature_names=list(X_rep2.columns)))


--- RESULTADOS COMPARATIVOS ---
                 Métrica Rep. 1: Z-Score Rep. 2: Binning
    Exactitud (Accuracy) 0.9313 ± 0.0202 0.9600 ± 0.0147
   Precisión (Precision)          0.9154          0.9389
   Sensibilidad (Recall)          0.8838          0.9459
Puntuación F1 (F1-Score)          0.8986          0.9418

--- REGLAS DE DECISIÓN GENERADAS (Representación 2: Binning) ---
|--- Uniformity_of_cell_size <= 0.50
|   |--- Bare_nuclei <= 0.50
|   |   |--- Normal_nucleoli <= 0.50
|   |   |   |--- Clump_thickness <= 1.50
|   |   |   |   |--- class: 0
|   |   |   |--- Clump_thickness >  1.50
|   |   |   |   |--- class: 0
|   |   |--- Normal_nucleoli >  0.50
|   |   |   |--- Uniformity_of_cell_shape <= 0.50
|   |   |   |   |--- class: 0
|   |   |   |--- Uniformity_of_cell_shape >  0.50
|   |   |   |   |--- class: 1
|   |--- Bare_nuclei >  0.50
|   |   |--- Clump_thickness <= 0.50
|   |   |   |--- Uniformity_of_cell_shape <= 0.50
|   |   |   |   |--- class: 0
|   |   |   |--- Uniformity_